# 🎮 Steam Game Recommender

Content-based recommendation system built on Steam Games Dataset 2025.

**Approach 1:** TF-IDF + Cosine Similarity (Content-Based Filtering)  
**Approach 2:** LightFM (Hybrid: Collaborative + Content)  
**Author:** ete9nal

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import ast
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k
import scipy.sparse as sp
import warnings

sys.path.append(os.path.abspath('..'))
from src.recommender import get_recommendations

warnings.filterwarnings('ignore')
print('All imports OK')

ModuleNotFoundError: No module named 'lightfm'

## 2. Data Loading

In [ ]:
df = pd.read_csv('../data/games_march2025_full.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 3. EDA

Quick look at key columns: review scores, genres distribution, missing values.

In [ ]:
# Check missing values in key columns
cols = ['name', 'genres', 'tags', 'about_the_game', 'pct_pos_total', 'num_reviews_total']
print('Missing values:')
print(df[cols].isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Positive review % distribution (exclude -1 which means no reviews)
sns.histplot(df[df['pct_pos_total'] > 0]['pct_pos_total'], bins=50, ax=axes[0])
axes[0].set_title('Positive Review % Distribution')
axes[0].set_xlabel('Positive %')

# Review count distribution (log scale)
sns.histplot(df[df['num_reviews_total'] > 0]['num_reviews_total'],
             log_scale=True, bins=50, ax=axes[1])
axes[1].set_title('Review Count Distribution (log scale)')
axes[1].set_xlabel('Number of Reviews')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 genres
all_genres = []
for g in df['genres'].dropna():
    try:
        all_genres.extend(ast.literal_eval(g))
    except:
        pass

pd.Series(all_genres).value_counts().head(10).plot(kind='bar', figsize=(10, 4))
plt.title('Top 10 Genres')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Preprocessing

- Remove games without description, genres, or tags
- Remove Playtest and Demo games (they have no real content)
- Parse genres/tags from string representation to plain text for TF-IDF

In [ ]:
# Drop rows with missing required fields
df_filtered = df[
    df['about_the_game'].notna() &
    df['genres'].notna() &
    df['tags'].notna() &
    df['name'].notna()
].copy()

# Remove Playtest and Demo games - they have no real content
df_filtered = df_filtered[
    ~df_filtered['name'].str.contains('Playtest|Demo', case=False, na=False)
].reset_index(drop=True)

print(f'Original:         {df.shape[0]:,} games')
print(f'After filtering:  {df_filtered.shape[0]:,} games')

In [ ]:
# Helper: parse list/dict stored as string -> plain text
def parse_to_text(val):
    if not isinstance(val, str) or val.strip() == '':
        return ''
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, dict):   # tags are {'Tag': count, ...}
            return ' '.join(parsed.keys())
        if isinstance(parsed, list):   # genres are ['Action', ...]
            return ' '.join(str(x) for x in parsed)
    except:
        pass
    return val

# Build combined_text for TF-IDF
# Genres and tags are repeated twice to give them higher weight vs description
df_filtered['genres_text']     = df_filtered['genres'].apply(parse_to_text)
df_filtered['tags_text']       = df_filtered['tags'].apply(parse_to_text)
df_filtered['categories_text'] = df_filtered['categories'].apply(parse_to_text)

df_filtered['combined_text'] = (
    df_filtered['genres_text']     + ' ' +
    df_filtered['genres_text']     + ' ' +   # repeated for weight
    df_filtered['tags_text']       + ' ' +
    df_filtered['tags_text']       + ' ' +   # repeated for weight
    df_filtered['categories_text'] + ' ' +
    df_filtered['about_the_game'].fillna('')
)

print('Combined text sample:')
print(df_filtered['combined_text'].iloc[0][:200])

## 5. Content-Based Filtering (TF-IDF + Cosine Similarity)

We vectorize each game's `combined_text` with TF-IDF, then compute cosine similarity.
To avoid RAM issues (~60 GB for full matrix), we compute similarity in batches.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95
)

X = vectorizer.fit_transform(df_filtered['combined_text'])
print(f'TF-IDF matrix shape: {X.shape}')

In [ ]:
# Compute top-50 similar games for each game using batches
# Full cosine_similarity(X) would need ~60 GB RAM - batching solves this
top_k = 50
batch_size = 1000
n = X.shape[0]
top_indices = np.zeros((n, top_k), dtype=np.int32)

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    batch_sim = cosine_similarity(X[start:end], X)          # (batch, n)
    batch_top = np.argsort(batch_sim, axis=1)[:, ::-1][:, :top_k]  # descending top-k
    top_indices[start:end] = batch_top
    if start % 10000 == 0:
        print(f'  {start}/{n} done...')

print(f'top_indices shape: {top_indices.shape}')

In [ ]:
# Save artifacts for Streamlit app
np.save('../data/top_indices.npy', top_indices)

df_filtered[[
    'name', 'genres', 'tags', 'categories',
    'positive', 'negative', 'header_image', 'price',
    'pct_pos_total', 'num_reviews_total'
]].reset_index(drop=True).to_parquet('../data/games_cleaned.parquet', index=False)

# Reload df to use the clean version everywhere below
df = pd.read_parquet('../data/games_cleaned.parquet')
print(f'Saved {len(df):,} games')

In [ ]:
# Quick sanity check - do recommendations make sense?
test_games = ['Counter-Strike 2', 'Dota 2', 'Grand Theft Auto V Legacy']

for game in test_games:
    recs = get_recommendations(game, df, top_indices, top_n=5)
    print(f'\n{game}:')
    for i, r in enumerate(recs, 1):
        print(f'  {i}. {r}')

## 6. Evaluation

Since we have no explicit user-item interactions, we define **ground truth** as:
- Same genre or tag overlap with the query game
- `pct_pos_total >= 75%` (well-rated)
- `num_reviews_total >= 50` (enough votes to trust the rating)

Metrics:
- **Precision@K** — what fraction of top-K recommendations are relevant
- **Recall@K** — what fraction of all relevant games we caught in top-K
- **NDCG@K** — like Precision but rewards relevant items ranked higher
- **Coverage@K** — what fraction of the catalog the model ever recommends

In [ ]:
def evaluate_recommendations(game_title, df, top_indices_matrix, top_n=5):
    match = df[df['name'].str.contains(game_title, case=False, na=False)]
    if match.empty:
        return None, None, None
    match = match.iloc[0]

    # Parse genres and tags of the query game
    try:
        genres = set(ast.literal_eval(match['genres']))
    except:
        genres = set()

    try:
        parsed_tags = ast.literal_eval(match['tags'])
        tags = set(parsed_tags.keys()) if isinstance(parsed_tags, dict) else set(parsed_tags)
    except:
        tags = set()

    # Build ground truth: games with genre/tag overlap + good reviews
    def genre_overlap(val):
        try: return bool(set(ast.literal_eval(val)) & genres)
        except: return False

    def tag_overlap(val):
        try:
            p = ast.literal_eval(val)
            return bool((set(p.keys()) if isinstance(p, dict) else set(p)) & tags)
        except: return False

    ground_truth = set(df[
        (df['pct_pos_total'] >= 75) &
        (df['num_reviews_total'] >= 50) &
        (df['genres'].apply(genre_overlap) | df['tags'].apply(tag_overlap))
    ]['name'])

    if not ground_truth:
        return 0, 0, 0

    recs = list(get_recommendations(game_title, df, top_indices_matrix, top_n=top_n))
    hits = len(set(recs) & ground_truth)

    precision = hits / top_n
    recall    = hits / len(ground_truth)
    dcg  = sum(1 / np.log2(i + 2) for i, g in enumerate(recs) if g in ground_truth)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(ground_truth), top_n)))
    ndcg = dcg / idcg if idcg > 0 else 0

    return precision, recall, ndcg


def calculate_coverage(top_indices_matrix, total_items, top_n=5):
    unique = set(top_indices_matrix[:, :top_n].flatten())
    return len(unique) / total_items

In [ ]:
test_games = ['Counter-Strike 2', 'Dota 2', 'Grand Theft Auto V Legacy']

for game in test_games:
    p, r, n = evaluate_recommendations(game, df, top_indices, top_n=5)
    if p is not None:
        print(f'{game}:')
        print(f'  Precision@5: {p:.2f}  |  Recall@5: {r:.4f}  |  NDCG@5: {n:.2f}')

coverage = calculate_coverage(top_indices, len(df), top_n=5)
print(f'\nCoverage@5: {coverage:.2%}')

## 7. LightFM — Hybrid Recommender

LightFM combines **collaborative filtering** (who likes what) with **content features** (genres/tags).
It's especially useful for the **cold-start problem**: recommending new games with no reviews yet.

We simulate user-item interactions using review data:
- `positive` reviews → implicit positive feedback
- `estimated_owners` → proxy for user base size

Since we don't have real user data, we build a **game-genre interaction matrix** instead:
each game interacts with its genres as if genres were 'users who liked this'.

In [ ]:
# Reload full df for LightFM (we need genres column as lists)
df_lfm = pd.read_parquet('../data/games_cleaned.parquet').reset_index(drop=True)

# Parse genres into lists
def safe_parse_list(val):
    try:
        return ast.literal_eval(val)
    except:
        return []

df_lfm['genres_list'] = df_lfm['genres'].apply(safe_parse_list)
df_lfm['tags_list']   = df_lfm['tags'].apply(
    lambda v: list(ast.literal_eval(v).keys()) if isinstance(v, str) else []
)

print(f'Games: {len(df_lfm)}')
print(f'Example genres: {df_lfm["genres_list"].iloc[0]}')
print(f'Example tags:   {df_lfm["tags_list"].iloc[0][:5]}')

In [ ]:
# Build LightFM Dataset
# 'users' = unique genres (we treat genre as a user proxy)
# 'items' = games

all_genres = set()
all_tags   = set()
for _, row in df_lfm.iterrows():
    all_genres.update(row['genres_list'])
    all_tags.update(row['tags_list'])

dataset = Dataset()
dataset.fit(
    users=list(all_genres),          # genres as users
    items=df_lfm['name'].tolist(),   # games as items
    item_features=list(all_tags)     # tags as item features
)

print(f'Unique genres (users): {len(all_genres)}')
print(f'Unique tags (features): {len(all_tags)}')

In [ ]:
# Build interaction matrix: genre -> game
# If a game belongs to a genre, that's a positive interaction
(interactions, weights) = dataset.build_interactions(
    [
        (genre, row['name'])
        for _, row in df_lfm.iterrows()
        for genre in row['genres_list']
    ]
)

print(f'Interaction matrix shape: {interactions.shape}')
print(f'Non-zero interactions: {interactions.nnz}')

In [ ]:
# Build item features matrix from tags
item_features = dataset.build_item_features(
    [
        (row['name'], row['tags_list'][:20])  # top 20 tags per game
        for _, row in df_lfm.iterrows()
    ]
)

print(f'Item features shape: {item_features.shape}')

In [ ]:
# Train LightFM with WARP loss
# WARP = Weighted Approximate-Rank Pairwise - optimizes for ranking (good for top-K)
model = LightFM(loss='warp', no_components=64, random_state=42)
model.fit(
    interactions,
    item_features=item_features,
    epochs=20,
    num_threads=4,
    verbose=True
)

In [ ]:
# Get LightFM recommendations for a genre
# i.e. 'what games would someone who likes Action games enjoy?'

_, item_id_map = dataset.mapping()[2], dataset.mapping()[2]
user_id_map = dataset.mapping()[0]
item_id_map = dataset.mapping()[2]

def get_lightfm_recs(genre, model, dataset, df_lfm, item_features, top_n=10):
    user_id_map, _, item_id_map, _ = dataset.mapping()

    if genre not in user_id_map:
        print(f'Genre "{genre}" not found')
        return []

    user_idx = user_id_map[genre]
    n_items  = len(item_id_map)

    # Score all items for this user (genre)
    scores = model.predict(
        user_ids=user_idx,
        item_ids=np.arange(n_items),
        item_features=item_features
    )

    # Get top-N item indices
    top_item_ids = np.argsort(-scores)[:top_n]

    # Map back to game names
    reverse_map = {v: k for k, v in item_id_map.items()}
    return [reverse_map[i] for i in top_item_ids]


# Test
for genre in ['Action', 'RPG', 'Strategy']:
    recs = get_lightfm_recs(genre, model, dataset, df_lfm, item_features, top_n=5)
    print(f'\nTop games for genre "{genre}":')
    for i, r in enumerate(recs, 1):
        print(f'  {i}. {r}')

In [ ]:
# LightFM Precision@K and Recall@K evaluation
train_precision = precision_at_k(model, interactions, item_features=item_features, k=5).mean()
train_recall    = recall_at_k(model, interactions, item_features=item_features, k=5).mean()

print(f'LightFM Precision@5: {train_precision:.4f}')
print(f'LightFM Recall@5:    {train_recall:.4f}')

## 8. Summary

| Model | Precision@5 | NDCG@5 | Coverage@5 | Cold Start |
|-------|------------|--------|------------|------------|
| TF-IDF + Cosine Similarity | ~0.20-0.40 | ~0.34-0.49 | ~99% | ❌ |
| LightFM (WARP) | see above | — | — | ✅ |

**Key takeaways:**
- Content-based model works well and covers almost all catalog
- LightFM handles cold start by using genre/tag features even for new games
- LightFM in this setup answers: *'what games fit a given genre?'*
- For production: combine both (ensemble / fallback strategy)